# Dalia: entrenamiento del modelo (tomate)

Este cuaderno reúne **todas las celdas del proyecto, en el orden en que se ejecutan**. Cada celda de código va precedida de una celda de texto que explica qué es y para qué sirve.

**Antes de empezar**
- Activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
- Colab borra archivos y variables al cerrarse. Descarga siempre tus modelos (`.pth`) apenas se entrenen.

**Partes**
- **Parte 0:** Preparación (GPU, descarga y revisión de los datasets).
- **Parte 1:** Modelo v1 (referencia, opcional).
- **Parte 2:** Modelo v2a (PlantVillage con aumentos fuertes, opcional).
- **Parte 3:** Probar tus propias fotos.
- **Parte 4:** Modelo v2b (PlantVillage + PlantDoc).
- **Parte 5:** Guardar tu trabajo.

**Atajo si solo quieres llegar a la v2b:** ejecuta la Parte 0, la celda 2.1 (datos), la celda 3.2 (solo si tienes un `.pth` para la línea base) y luego la Parte 4.

**Celdas manuales:** las que suben archivos (`files.upload()`) están comentadas con `#`, para que *Ejecutar todo* no se quede esperando. Quita el `#` solo cuando quieras usarlas.

## Parte 0. Preparación

### 0.1 Comprobar que la GPU está activa
Muestra si Colab tiene GPU. Sin GPU el entrenamiento es muy lento. Si dice `False`, actívala en *Entorno de ejecución → Cambiar tipo de entorno → GPU* y vuelve a ejecutar desde aquí.

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tarjeta:", torch.cuda.get_device_name(0))
else:
    print("Activa la GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU")

### 0.2 Descargar los datasets (PlantVillage y PlantDoc)
Clona los dos repositorios de imágenes. Solo descarga si la carpeta aún no existe, así que puedes ejecutarla varias veces. PlantVillage pesa unos 2 GiB y puede tardar unos minutos. Revisa el `LICENSE.txt` de cada dataset y cítalos en tu documentación.

In [ ]:
!test -d PlantVillage-Dataset || git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git
!test -d PlantDoc-Dataset || git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git

### 0.3 Verificar que PlantVillage se descargó completo
Cuenta las carpetas de tomate en PlantVillage. **Debe imprimir 10.** Si imprime menos, borra la carpeta (`!rm -rf PlantVillage-Dataset`) y vuelve a ejecutar la celda 0.2.

In [ ]:
!ls PlantVillage-Dataset/raw/color | grep -c Tomato

### 0.4 Inspeccionar las carpetas de PlantDoc
Lista el contenido de PlantDoc y sus clases de tomate. Sirve para comprobar los nombres de carpeta que usa el mapeo de clases de la celda 4.1.

In [ ]:
!ls PlantDoc-Dataset
!ls PlantDoc-Dataset/train | grep -i tomato

### 0.5 Verificar que "Tomato leaf" (PlantDoc) contiene hojas sanas
Muestra 4 imágenes al azar de la carpeta `Tomato leaf`. Deberías ver hojas sanas, porque el mapeo la trata como la clase `healthy`. Si ves hojas enfermas, avísame antes de seguir.

In [ ]:
import glob, random
from PIL import Image
from IPython.display import display

fotos = glob.glob("PlantDoc-Dataset/train/Tomato leaf/*")
print(len(fotos), "imágenes")
for f in random.sample(fotos, 4):
    im = Image.open(f).convert("RGB")
    im.thumbnail((300, 300))
    display(im)

## Parte 1. Modelo v1 (referencia, opcional)

### 1.1 Entrenar el modelo v1 (PlantVillage, tomate, 3 épocas)
Primer entrenamiento: MobileNetV2 con transfer learning sobre las 10 clases de tomate de PlantVillage. Es la versión corregida del script original: tiene validación real (sin augmentation), etiquetas de 0 a 9 y guarda los nombres de las clases junto con los pesos. Guarda `dalia_tomato_mobilenetv2.pth`. Si no quieres reproducir la v1, puedes saltarte esta parte.

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

DATASET_DIR = "PlantVillage-Dataset/raw/color"
PLANT = "Tomato"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([  # sin augmentation
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

ds_train = datasets.ImageFolder(DATASET_DIR, transform=train_tf)
ds_val = datasets.ImageFolder(DATASET_DIR, transform=val_tf)

# Solo clases del cultivo elegido, con etiquetas reindexadas 0..N-1
class_names = [c for c in ds_train.classes if PLANT in c]
old2new = {ds_train.class_to_idx[c]: i for i, c in enumerate(class_names)}
ds_train.target_transform = old2new.__getitem__
ds_val.target_transform = old2new.__getitem__

selected = [i for i, (_, y) in enumerate(ds_train.samples) if y in old2new]
random.shuffle(selected)
cut = int(0.8 * len(selected))
train_idx, val_idx = selected[:cut], selected[cut:]

train_loader = DataLoader(Subset(ds_train, train_idx), batch_size=32,
                          shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(ds_val, val_idx), batch_size=32,
                        shuffle=False, num_workers=2)

print(f"Clases ({len(class_names)}):", *class_names, sep="\n  - ")
print(f"Entrenamiento: {len(train_idx)} | Validación: {len(val_idx)}")

# Modelo
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
for p in model.features.parameters():
    p.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    if train:
        model.features.eval()  # mantiene fijo el BatchNorm del backbone
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

EPOCHS = 3
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"Época {epoch+1}/{EPOCHS} | train: {tr_loss:.4f} / {tr_acc:.4f} "
          f"| val: {va_loss:.4f} / {va_acc:.4f}")

# Guardar pesos + nombres de clases
torch.save({
    "state_dict": model.state_dict(),
    "class_names": class_names,
    "arch": "mobilenet_v2",
    "input_size": 224,
}, "dalia_tomato_mobilenetv2.pth")
print("Guardado: dalia_tomato_mobilenetv2.pth")

### 1.2 Evaluar el modelo v1 por clase
Muestra, para cada enfermedad, su *precision* (cuando el modelo dice "X", cuántas veces acierta) y su *recall* (de las hojas que de verdad tenían X, cuántas encontró), más la matriz de confusión (filas = clase real, columnas = clase predicha). Sirve para ver qué clases se confunden entre sí. Usa el modelo y la validación de la celda 1.1.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

short = [c.replace("Tomato___", "") for c in class_names]
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_loader:
        out = model(x.to(device))
        y_pred.extend(out.argmax(1).cpu().tolist())
        y_true.extend(y.tolist())

print(classification_report(y_true, y_pred, target_names=short, digits=3))
print(confusion_matrix(y_true, y_pred))  # filas = real, columnas = predicho

### 1.3 Descargar el modelo v1
Descarga `dalia_tomato_mobilenetv2.pth` a tu computadora. Guárdalo en tu carpeta `dalia/ml/`. Si el navegador pregunta si permite descargas, acepta.

In [ ]:
from google.colab import files
files.download("dalia_tomato_mobilenetv2.pth")

## Parte 2. Modelo v2a (PlantVillage con aumentos fuertes)

### 2.1 Preparar los datos de la v2a
Prepara las imágenes de tomate de PlantVillage con aumentos más fuertes (recortes aleatorios, cambios de brillo y color, borrado aleatorio), el reparto 80/20 y los pesos por clase (para que clases pequeñas como el virus del mosaico pesen más). **Esta celda también la necesita la v2b**, así que ejecútala aunque te saltes la v2a.

In [ ]:
import os, random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

DATASET_DIR = "PlantVillage-Dataset/raw/color"
PLANT = "Tomato"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

ds_train = datasets.ImageFolder(DATASET_DIR, transform=train_tf)
ds_val = datasets.ImageFolder(DATASET_DIR, transform=val_tf)

class_names = [c for c in ds_train.classes if PLANT in c]
old2new = {ds_train.class_to_idx[c]: i for i, c in enumerate(class_names)}
ds_train.target_transform = old2new.__getitem__
ds_val.target_transform = old2new.__getitem__

selected = [i for i, (_, y) in enumerate(ds_train.samples) if y in old2new]
random.shuffle(selected)
cut = int(0.8 * len(selected))
train_idx, val_idx = selected[:cut], selected[cut:]

train_loader = DataLoader(Subset(ds_train, train_idx), batch_size=32,
                          shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(ds_val, val_idx), batch_size=32,
                        shuffle=False, num_workers=2)

# Pesos por clase (las clases con pocas imágenes pesan más)
counts = torch.zeros(len(class_names))
for i in train_idx:
    counts[old2new[ds_train.samples[i][1]]] += 1
class_weights = (counts.sum() / (len(class_names) * counts)).to(device)

print(f"Clases: {len(class_names)} | Entrenamiento: {len(train_idx)} | Validación: {len(val_idx)}")

### 2.2 Entrenar el modelo v2a
Ajusta, además de la capa final, los bloques finales de MobileNetV2 con una tasa de aprendizaje baja, durante 8 épocas con programador de tasa de aprendizaje y pesos por clase. Guarda el mejor modelo según la validación como `dalia_tomato_v2.pth`. Necesita las variables de la celda 2.1.

In [ ]:
UNFREEZE_FROM = 14   # bloques finales de MobileNetV2 que se ajustan
EPOCHS = 8
USE_CLASS_WEIGHTS = True

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
for p in model.features.parameters():
    p.requires_grad = False
for p in model.features[UNFREEZE_FROM:].parameters():
    p.requires_grad = True
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTS else None)
optimizer = optim.Adam([
    {"params": model.features[UNFREEZE_FROM:].parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    if train:
        model.features[:UNFREEZE_FROM].eval()  # BatchNorm fijo en las capas congeladas
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

best = 0.0
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"Época {epoch+1}/{EPOCHS} | train: {tr_loss:.4f} / {tr_acc:.4f} "
          f"| val: {va_loss:.4f} / {va_acc:.4f}")
    if va_acc > best:
        best = va_acc
        torch.save({
            "state_dict": model.state_dict(),
            "class_names": class_names,
            "arch": "mobilenet_v2",
            "input_size": 224,
            "version": "v2a",
        }, "dalia_tomato_v2.pth")
        print("  -> guardado dalia_tomato_v2.pth")

### 2.3 Descargar el modelo v2a
Descarga `dalia_tomato_v2.pth` a tu computadora (carpeta `dalia/ml/`).

In [ ]:
from google.colab import files
files.download("dalia_tomato_v2.pth")

## Parte 3. Probar tus propias fotos

### 3.0 Comprobar qué modelos y fotos hay en el disco
Lista los `.pth` y las fotos que Colab tiene ahora mismo. Si el navegador le agregó un `(1)` al nombre de un modelo que subiste, renómbralo con el comando de la segunda línea (quita el `#` y ajusta el nombre).

In [ ]:
!ls -lh *.pth
!ls *.jp*g *.webp
# !mv "dalia_tomato_v2 (1).pth" dalia_tomato_v2.pth

### 3.1 Subir un modelo o fotos (manual)
Abre el selector de archivos de Colab. Úsala para subir un `.pth` guardado (si abriste una sesión nueva) o tus fotos de prueba. Está comentada para que *Ejecutar todo* no se detenga aquí: quita el `#` de la última línea cuando la necesites. Se guarda en `_` para que Colab no imprima los bytes del archivo.

In [ ]:
from google.colab import files
# Quita el "#" de la línea siguiente cuando quieras subir archivos
# _ = files.upload()

### 3.2 Cargar un modelo y predecir las fotos del disco
Define las funciones para cargar un modelo guardado y predecir fotos, carga el modelo indicado en `CKPT` (por defecto el v2a) y predice todas las fotos `.jpg`, `.jpeg` y `.webp` que haya en el disco, mostrando las 3 causas más probables con su porcentaje. Omite duplicados como `foto (1).jpg`. Deja el modelo cargado en la variable `net`, que usa la línea base de la celda 4.1.

In [ ]:
import glob
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def cargar_modelo(ruta):
    ckpt = torch.load(ruta, map_location=device, weights_only=True)
    nombres = ckpt["class_names"]
    m = models.mobilenet_v2(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(nombres))
    m.load_state_dict(ckpt["state_dict"])
    return m.to(device).eval(), nombres

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(path, topk=3):
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(net(x), dim=1)[0]
    conf, idx = probs.topk(topk)
    for c, i in zip(conf.tolist(), idx.tolist()):
        print(f"  {short[i]}: {c*100:.1f}%")

def fotos_en_disco():
    nombres = sorted(glob.glob("*.jp*g") + glob.glob("*.webp"))
    return [n for n in nombres if "(" not in n]   # omite duplicados como "tomate (1).jpg"

CKPT = "dalia_tomato_v2.pth"   # cambia aquí el modelo que quieres probar
net, class_names = cargar_modelo(CKPT)
short = [c.replace("Tomato___", "") for c in class_names]
print("Modelo cargado:", CKPT)

for name in fotos_en_disco():
    print(name)
    predict(name)

### 3.3 Probar fotos nuevas sin repetir las anteriores (manual)
Sube fotos nuevas y predice solo esas con el modelo cargado en `net`. Antes de subir cada foto, anota qué tiene realmente (sana, manchas, falta de agua, etc.) para comparar después. Está comentada: quita el `#` de las cuatro líneas cuando la necesites. Requiere haber ejecutado la celda 3.2.

In [ ]:
# from google.colab import files
# nuevas = files.upload()
# for nombre in nuevas:
#     print(nombre)
#     predict(nombre)

### 3.4 Comparar tres formas de preparar la imagen
Prueba con cada foto tres preprocesados distintos (estirar a 224x224, recorte central y cuadrado con borde gris) y muestra la predicción principal y la probabilidad de `healthy`. Sirve para ver si el modelo es frágil al preprocesado. En nuestras pruebas, `healthy` seguía cerca de 0% en las tres variantes. Requiere la celda 3.2.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
norm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def v_estirar(img):   # lo que usamos hasta ahora
    return norm(transforms.Resize((224, 224))(img))

def v_recorte(img):   # lado corto a 256 y recorte central
    return norm(transforms.CenterCrop(224)(transforms.Resize(256)(img)))

def v_relleno(img):   # cuadrado con borde gris, sin deformar
    w, h = img.size
    s = max(w, h)
    canvas = Image.new("RGB", (s, s), (128, 128, 128))
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return norm(transforms.Resize((224, 224))(canvas))

def resumen(x):
    i_healthy = class_names.index("Tomato___healthy")
    with torch.no_grad():
        p = torch.softmax(net(x.unsqueeze(0).to(device)), dim=1)[0]
    c, i = p.max(0)
    return f"{short[i]} {c.item()*100:.0f}% | healthy {p[i_healthy].item()*100:.1f}%"

def comparar_preprocesado():
    for name in fotos_en_disco():
        img = Image.open(name).convert("RGB")
        print(name)
        print("  estirar:", resumen(v_estirar(img)))
        print("  recorte:", resumen(v_recorte(img)))
        print("  relleno:", resumen(v_relleno(img)))

comparar_preprocesado()

## Parte 4. Modelo v2b (PlantVillage + PlantDoc)

### 4.1 Mezclar PlantVillage con PlantDoc
Mapea las clases de PlantDoc a las 10 de PlantVillage (`Tomato leaf` se trata como `healthy` y `yellow virus` como el virus del rizado amarillo: son suposiciones; `Target_Spot` no tiene equivalente en PlantDoc). Repite las imágenes de PlantDoc para que pesen cerca del 30% del tamaño de PlantVillage y recalcula los pesos por clase. El test de PlantDoc se usa como medida de fotos reales. Si tienes un modelo cargado en `net` (celda 3.2), calcula su resultado en PlantDoc como línea base. **Requiere la celda 2.1.**

In [ ]:
from torch.utils.data import ConcatDataset

PD_DIR = "PlantDoc-Dataset"
PD_MAP = {
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
}
assert all(v in class_names for v in PD_MAP.values()), "Revisa los nombres de clase"

def make_pd(split, tf):
    ds = datasets.ImageFolder(f"{PD_DIR}/{split}", transform=tf)
    m = {ds.class_to_idx[k]: class_names.index(v)
         for k, v in PD_MAP.items() if k in ds.class_to_idx}
    ds.target_transform = m.__getitem__
    keep = [i for i, (_, y) in enumerate(ds.samples) if y in m]
    return ds, keep, m

pd_train_ds, pd_train_keep, pd_train_map = make_pd("train", train_tf)
pd_test_ds, pd_test_keep, _ = make_pd("test", val_tf)
pd_train = Subset(pd_train_ds, pd_train_keep)
pd_test = Subset(pd_test_ds, pd_test_keep)
assert len(pd_train) > 0 and len(pd_test) > 0
print("PlantDoc tomate -> train:", len(pd_train), "| test:", len(pd_test))

# PlantDoc es pequeño: se repite para que pese ~30% del tamaño de PlantVillage
pv_train = Subset(ds_train, train_idx)
REPEAT = max(1, min(10, round(0.3 * len(pv_train) / len(pd_train))))
train_ds = ConcatDataset([pv_train] + [pd_train] * REPEAT)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
pd_test_loader = DataLoader(pd_test, batch_size=32, shuffle=False, num_workers=2)
print(f"PlantDoc se repite x{REPEAT} | imágenes por época: {len(train_ds)}")

# Pesos por clase con los datos combinados
counts = torch.zeros(len(class_names))
for i in train_idx:
    counts[old2new[ds_train.samples[i][1]]] += 1
for i in pd_train_keep:
    counts[pd_train_map[pd_train_ds.samples[i][1]]] += REPEAT
class_weights = (counts.sum() / (len(class_names) * counts)).to(device)

def evaluate(m, loader):
    m.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            y_pred += m(x.to(device)).argmax(1).cpu().tolist()
            y_true += y.tolist()
    acc = sum(int(a == b) for a, b in zip(y_true, y_pred)) / len(y_true)
    return acc, y_true, y_pred

if "net" in globals():
    acc0, _, _ = evaluate(net, pd_test_loader)
    print(f"Línea base (modelo cargado en la celda 3.2) en PlantDoc (test): {acc0:.3f}")
else:
    print("Sin línea base: ejecuta antes la celda 3.2 si quieres compararla.")

### 4.2 Entrenar el modelo v2b
Entrena con PlantVillage + PlantDoc durante 8 épocas, con el mismo ajuste fino de la v2a. En cada época imprime dos números: la validación de PlantVillage y el test de PlantDoc (fotos reales). Guarda el modelo de la **última** época (no el mejor, para no elegir con el mismo test que reportamos) como `dalia_tomato_v2b.pth`, e imprime al final el reporte por clase sobre el test de PlantDoc. Fíjate sobre todo en el recall de `healthy`. Requiere la celda 4.1.

In [ ]:
from sklearn.metrics import classification_report

UNFREEZE_FROM = 14
EPOCHS = 8

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
for p in model.features.parameters():
    p.requires_grad = False
for p in model.features[UNFREEZE_FROM:].parameters():
    p.requires_grad = True
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam([
    {"params": model.features[UNFREEZE_FROM:].parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_epoch():
    model.train()
    model.features[:UNFREEZE_FROM].eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch()
    pv_acc, _, _ = evaluate(model, val_loader)
    pd_acc, _, _ = evaluate(model, pd_test_loader)
    scheduler.step()
    print(f"Época {epoch+1}/{EPOCHS} | train: {tr_loss:.4f} / {tr_acc:.4f} "
          f"| val PlantVillage: {pv_acc:.4f} | test PlantDoc: {pd_acc:.4f}")

torch.save({
    "state_dict": model.state_dict(),
    "class_names": class_names,
    "arch": "mobilenet_v2",
    "input_size": 224,
    "version": "v2b",
}, "dalia_tomato_v2b.pth")
print("Guardado: dalia_tomato_v2b.pth")

_, yt, yp = evaluate(model, pd_test_loader)
labels = sorted(set(yt))
print(classification_report(yt, yp, labels=labels, digits=3, zero_division=0,
      target_names=[class_names[i].replace("Tomato___", "") for i in labels]))

### 4.3 Descargar el modelo v2b
Descarga `dalia_tomato_v2b.pth` apenas termine el entrenamiento y guárdalo en `dalia/ml/`.

In [ ]:
from google.colab import files
files.download("dalia_tomato_v2b.pth")

### 4.4 Probar tus fotos con el modelo v2b
Carga el modelo v2b, predice las fotos del disco y vuelve a comparar los tres preprocesados. La comparación clave son las fotos sanas: ¿ahora `healthy` deja de estar cerca de 0%? Requiere haber ejecutado antes las celdas 3.2 y 3.4 (definen las funciones que usa) y tener tus fotos en el disco (celda 3.1).

In [ ]:
net, class_names = cargar_modelo("dalia_tomato_v2b.pth")
short = [c.replace("Tomato___", "") for c in class_names]
print("Modelo cargado: dalia_tomato_v2b.pth")

for name in fotos_en_disco():
    print(name)
    predict(name)

print("\n--- Comparación de preprocesados ---")
comparar_preprocesado()

## Parte 5. Guardar tu trabajo

### 5.1 Descargar todos los modelos que existan
Descarga a tu computadora todos los `.pth` que haya en el disco de Colab. Ejecútala antes de cerrar la pestaña. Si el navegador pregunta si permite varias descargas, acepta. Después descarga también este cuaderno (*Archivo → Descargar → Descargar .ipynb*) y guarda todo en `dalia/ml/`.

In [ ]:
import os
from google.colab import files
for f in ["dalia_tomato_mobilenetv2.pth", "dalia_tomato_v2.pth", "dalia_tomato_v2b.pth"]:
    if os.path.exists(f):
        files.download(f)

## Notas del proyecto

- **Alcance del MVP:** tomate en maceta con PlantVillage. Romero, nochebuena y enredaderas quedan en espera.
- **Modelo:** MobileNetV2 con transfer learning (PyTorch), 10 clases de tomate.
- **v1:** validación 0.905. **v2a:** 0.976. Ambas fallan con fotos reales sanas (`healthy` casi 0%, con Late_blight como respuesta por defecto).
- **Hipótesis descartadas:** ni el fondo ni la deformación de la imagen explican el fallo por sí solos.
- **v2b:** mezcla PlantDoc para añadir hojas reales. Se mide con el test de PlantDoc y con tus 10 fotos (que no se usan para entrenar).
- **Pendiente:** capa Domain, API, app Flutter y YOLO para plagas en una versión futura.